# Eksperimen Machine Learning: Credit Card Default Prediction
**Nama Siswa**: Steven Gianmarg Haposan Siahaan  
**Dataset**: UCI Default of Credit Card Clients  
**Tujuan**: Memprediksi apakah seorang nasabah akan gagal bayar bulan berikutnya (klasifikasi biner)

---


## 1. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import joblib

print("✅ Libraries loaded successfully")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   sklearn : {__import__('sklearn').__version__}")

## 2. Data Loading

Load dataset UCI Default of Credit Card Clients.  
Dataset berisi informasi 30.000 nasabah kartu kredit di Taiwan (April–September 2005).


In [ ]:
RAW_PATH = os.path.join('..', 'creditcard_raw.xls')

df_raw = pd.read_excel(RAW_PATH, header=1)

print(f"✅ Dataset loaded successfully")
print(f"   Shape   : {df_raw.shape}")
print(f"   Rows    : {df_raw.shape[0]:,}")
print(f"   Columns : {df_raw.shape[1]}")

In [ ]:
df_raw.head(5)

In [ ]:
print("=== INFORMASI KOLOM ===")
print()
print("ID              : ID unik nasabah")
print("LIMIT_BAL       : Batas kredit (NT dollar)")
print("SEX             : Jenis kelamin (1=laki-laki, 2=perempuan)")
print("EDUCATION       : Tingkat pendidikan (1=graduate school, 2=university, 3=high school, 4=others)")
print("MARRIAGE        : Status pernikahan (1=married, 2=single, 3=others)")
print("AGE             : Usia (tahun)")
print("PAY_0..PAY_6    : Status pembayaran Sep-Apr 2005 (-2=no consumption, -1=paid duly, 0..8=months delayed)")
print("BILL_AMT1..6    : Tagihan kartu kredit Sep-Apr 2005 (NT dollar)")
print("PAY_AMT1..6     : Jumlah pembayaran Sep-Apr 2005 (NT dollar)")
print("TARGET          : default payment next month (1=yes, 0=no)")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("=== INFORMASI DASAR DATASET ===")
df_raw.info()

In [ ]:
print("=== STATISTIK DESKRIPTIF ===")
df_raw.describe().T

In [ ]:
print("=== MISSING VALUES ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(missing_df[missing_df['count'] > 0] if missing_df['count'].sum() > 0 else "✅ Tidak ada missing values!")
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
print("=== DISTRIBUSI TARGET ===")
target_col = 'default payment next month'
target_counts = df_raw[target_col].value_counts()
target_pct = df_raw[target_col].value_counts(normalize=True) * 100

print(f"Class 0 (Tidak default): {target_counts[0]:,} ({target_pct[0]:.1f}%)")
print(f"Class 1 (Default)       : {target_counts[1]:,} ({target_pct[1]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
axes[0].bar(['Tidak Default (0)', 'Default (1)'], target_counts.values,
            color=['#2ecc71', '#e74c3c'], alpha=0.85, edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribusi Target Variable', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Nasabah')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({target_pct.values[i]:.1f}%)',
                 ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['Tidak Default', 'Default'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, explode=(0, 0.05), shadow=True)
axes[1].set_title('Proporsi Default vs Non-Default', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join('creditcard_preprocessing', 'target_distribution.png'), 
            dpi=150, bbox_inches='tight') if os.path.exists('creditcard_preprocessing') else None
plt.show()

In [ ]:
num_features = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'PAY_AMT1']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df_raw[col], bins=50, color='#3498db', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i].set_title(f'Distribusi: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].axvline(df_raw[col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {df_raw[col].mean():,.0f}')
    axes[i].legend()

plt.suptitle('Distribusi Fitur Numerik Utama', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
cat_features = ['SEX', 'EDUCATION', 'MARRIAGE']
labels_map = {
    'SEX': {1: 'Laki-laki', 2: 'Perempuan'},
    'EDUCATION': {1: 'Graduate', 2: 'University', 3: 'High School', 4: 'Others', 5: 'Unk5', 6: 'Unk6', 0: 'Unk0'},
    'MARRIAGE': {0: 'Others', 1: 'Married', 2: 'Single', 3: 'Others'}
}
cat_colors = ['#9b59b6', '#e67e22', '#1abc9c']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(cat_features):
    counts = df_raw[col].value_counts().sort_index()
    axes[i].bar(counts.index.astype(str), counts.values, 
                color=cat_colors[i], alpha=0.85, edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'Distribusi: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Jumlah')
    for j, (idx, v) in enumerate(zip(counts.index, counts.values)):
        axes[i].text(j, v + 100, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df_numeric = df_raw.drop(columns=['ID'])
corr_matrix = df_numeric.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap — Credit Card Default Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

target_corr = corr_matrix['default payment next month'].abs().sort_values(ascending=False)
print("\n=== TOP 10 FITUR BERKORELASI DENGAN TARGET ===")
print(target_corr.head(11).to_string())

In [ ]:
pay0_default = df_raw.groupby('PAY_0')['default payment next month'].mean() * 100
print("Default rate berdasarkan PAY_0 (status pembayaran September 2005):")
print(pay0_default.round(1).to_string())

plt.figure(figsize=(12, 5))
bars = plt.bar(pay0_default.index, pay0_default.values,
               color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(pay0_default))),
               edgecolor='black', linewidth=0.5)
plt.title('Default Rate (%) berdasarkan Status Pembayaran PAY_0', fontsize=13, fontweight='bold')
plt.xlabel('PAY_0 (Status Pembayaran Sep 2005)')
plt.ylabel('Default Rate (%)')
plt.xticks(pay0_default.index)
for bar, val in zip(bars, pay0_default.values):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
df = df_raw.copy()
print(f"Dataset copied — shape: {df.shape}")

In [ ]:
df = df.drop(columns=['ID'])
df = df.rename(columns={'default payment next month': 'default'})
print(f"After drop ID & rename: {df.shape}")
print(f"Target column: 'default'")

In [ ]:
print("=== ANOMALOUS VALUES ===")
print(f"EDUCATION values before: {sorted(df['EDUCATION'].unique())}")
print(f"MARRIAGE values before : {sorted(df['MARRIAGE'].unique())}")

df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in pay_cols:
    df[col] = df[col].clip(lower=-2)

print(f"\nEDUCATION values after: {sorted(df['EDUCATION'].unique())}")
print(f"MARRIAGE values after : {sorted(df['MARRIAGE'].unique())}")
print("✅ Anomalous values handled")

In [ ]:
print("=== FEATURE ENGINEERING ===")
bill_cols    = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']
pay_amt_cols = ['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

df['AVG_BILL_AMT']      = df[bill_cols].mean(axis=1)
df['AVG_PAY_AMT']       = df[pay_amt_cols].mean(axis=1)
df['PAY_RATIO']         = (df['AVG_PAY_AMT'] / (df['AVG_BILL_AMT'] + 1)).clip(0, 10)
df['CREDIT_UTIL']       = (df['BILL_AMT1'] / (df['LIMIT_BAL'] + 1)).clip(0, 5)
df['LATE_PAYMENT_COUNT'] = (df[pay_cols] > 0).sum(axis=1)

print(f"Shape before FE: {df_raw.drop(columns=['ID']).shape}")
print(f"Shape after FE : {df.shape}")
print(f"New features   : AVG_BILL_AMT, AVG_PAY_AMT, PAY_RATIO, CREDIT_UTIL, LATE_PAYMENT_COUNT")

In [ ]:
print("=== ONE-HOT ENCODING ===")
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
print(f"Shape after OHE: {df.shape}")
print(f"Encoded columns: {[c for c in df.columns if any(c.startswith(p) for p in ['SEX_','EDUCATION_','MARRIAGE_'])]}")

In [ ]:
X = df.drop(columns=['default'])
y = df['default']

print("=== FEATURES & TARGET ===")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts().to_string())
print(f"\nClass imbalance ratio: {y.value_counts()[0]/y.value_counts()[1]:.2f}:1")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== TRAIN-TEST SPLIT ===")
print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")
print(f"\nTrain target distribution: {dict(y_train.value_counts())}")
print(f"Test  target distribution: {dict(y_test.value_counts())}")

In [ ]:
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
binary_prefixes = ('SEX_', 'EDUCATION_', 'MARRIAGE_')
numerical_cols = [c for c in numerical_cols if not c.startswith(binary_prefixes)]

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols]  = scaler.transform(X_test[numerical_cols])

print("=== FEATURE SCALING ===")
print(f"Scaled {len(numerical_cols)} numerical features using StandardScaler")
print("(fit on train only — no data leakage)")
print(f"\nX_train stats after scaling:")
print(X_train[numerical_cols[:3]].describe().loc[['mean','std']].round(3).to_string())

In [ ]:
OUTPUT_DIR = 'creditcard_preprocessing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(os.path.join(OUTPUT_DIR, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, 'X_test.csv'),  index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, 'y_train.csv'), index=False, header=True)
y_test.to_csv(os.path.join(OUTPUT_DIR, 'y_test.csv'),  index=False, header=True)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))

print("=== DATA TERSIMPAN ===")
for f in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  ✅ {f} ({size/1024:.1f} KB)")

In [ ]:
print("=" * 60)
print("  RINGKASAN PREPROCESSING")
print("=" * 60)
print(f"  Dataset awal    : {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom")
print(f"  Dataset akhir   : {df.shape[0]:,} baris × {df.shape[1]} kolom")
print(f"  Missing values  : 0")
print(f"  Fitur baru      : 5 (AVG_BILL_AMT, AVG_PAY_AMT, PAY_RATIO, CREDIT_UTIL, LATE_PAYMENT_COUNT)")
print(f"  OHE columns     : SEX, EDUCATION, MARRIAGE")
print(f"  Training set    : {X_train.shape}")
print(f"  Test set        : {X_test.shape}")
print(f"  Scaler          : StandardScaler (fit on train)")
print()
print("✅ Preprocessing selesai — data siap untuk pelatihan model!")